# Cramer-Rao bound analysis for eSOH identifiability

This notebook supports the sampling-frequency and SoC-window coverage analysis in the Supplementary Information. The CRB is computed for the seven-parameter eSOH model and propagated to diagnostic quantities used in the paper. For the figure panels, the reported set contains five transformed diagnostic parameters (`Cn_Si`, `Cn_Gr`, `Cn`, `Cp`, `LI`) together with the two silicon-OCP deformation parameters (`s_V`, `U_off`). The exported CSV files preserve the historical internal column names `a` and `b` for these two deformation parameters so downstream plotting scripts remain compatible. Stoichiometric limits (`x_n100`, `x_p100`) are retained in the exported tables for traceability and correspond to `x_n,100` and `x_p,100` in the manuscript. Transition SoC is not included in this CRB propagation because it is a derived metric and is evaluated separately in the C-rate study.

Two practical questions are addressed here. First, the sampling-frequency sweep estimates the minimum data requirement by showing how CRB-derived error bounds decrease as the number of voltage points increases over the 0--100% SoC window. The study adopts 8 points per SoC, or 800 total points, as the downsampled charge-trace resolution once all reported error bounds fall below approximately 5%. Second, the SoC-window sweep evaluates which partial charge windows remain informative when full-window diagnostic data are unavailable.


In [ ]:
import os
import pickle
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Return the repository root that contains the shared code and data folders."""
    for root in [start.resolve(), *start.resolve().parents]:
        if (root / "code" / "diagnostic_algorithm_lifetime_crate").exists() and (root / "data" / "cell_ocp").exists():
            return root
    raise FileNotFoundError("Could not locate repository root from the current working directory.")


PROJECT_ROOT = find_repo_root(Path.cwd())
CODE_DIR = PROJECT_ROOT / "code"
ANALYSIS_DIR = CODE_DIR / "error_bound_analysis"
OUTPUT_DIR = ANALYSIS_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
MPLCONFIG_DIR = Path("/tmp") / "managing_si_burnout_matplotlib"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR))

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.stats import t

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))


In [ ]:
from diagnostic_algorithm_lifetime_crate.user_functions import ocp_3


def OCP(X, Q, use_reconstructed_si_ocp=True):
    """Scalar full-cell OCP wrapper used by the CRB finite-difference model."""
    ocp, _, _ = ocp_3(X, Q, use_reconstructed_si_ocp=use_reconstructed_si_ocp)
    return float(np.asarray(ocp))


def compute_jacobian(OCP, res, Qdata, scale=0.03, use_reconstructed_si_ocp=True):
    """Compute voltage sensitivities used to form the Fisher information matrix."""
    res = np.asarray(res, dtype=float).copy()
    Qdata = np.asarray(Qdata, dtype=float)

    n_params = len(res)
    n_data = len(Qdata)
    J = np.zeros((n_data, n_params), dtype=float)

    Vfit = np.array(
        [OCP(res, Q, use_reconstructed_si_ocp=use_reconstructed_si_ocp) for Q in Qdata],
        dtype=float,
    )

    for k in range(n_params):
        delta = scale * abs(res[k]) if abs(res[k]) > 0 else 1e-6
        res_perturbed = res.copy()
        res_perturbed[k] += delta

        Vfit_perturbed = np.array(
            [OCP(res_perturbed, Q, use_reconstructed_si_ocp=use_reconstructed_si_ocp) for Q in Qdata],
            dtype=float,
        )
        J[:, k] = (Vfit_perturbed - Vfit) / delta

    return J


In [ ]:
# Internal aliases match saved CRB CSVs: x100 = x_n,100, y100 = x_p,100,
# a = s_V, and b = U_off in the manuscript notation.
BASE_NAMES = ["Cn_Si", "Cn_Gr", "x100", "Cp", "y100", "a", "b"]
PHI_NAMES = ["Cn_Si", "Cn_Gr", "Cp", "x_n100", "x_p100", "a", "b", "Cn", "LI"]


def crb_and_propagate(OCP, compute_jacobian, res, Qdata, sigma2, alpha=0.05, jac_kwargs=None):
    """Compute CRB and propagate uncertainty to the quantities reported in the paper.

    The base fit uses [Cn_Si, Cn_Gr, x100, Cp, y100, a, b], where
    x100/y100/a/b correspond to x_n,100/x_p,100/s_V/U_off. The exported
    propagated set includes stoichiometric limits for traceability, while the
    figure panels focus on Cn_Si, Cn_Gr, Cn, Cp, LI, s_V, and U_off. Total anode
    capacity Cn and lithium inventory LI are derived by first-order
    delta-method covariance propagation.
    """
    if jac_kwargs is None:
        jac_kwargs = {}

    J = compute_jacobian(OCP, res, Qdata, **jac_kwargs)
    FIM = (1.0 / sigma2) * (J.T @ J)
    try:
        CRB = np.linalg.inv(FIM)
    except np.linalg.LinAlgError:
        CRB = np.linalg.pinv(FIM)

    SE_base = np.sqrt(np.diag(CRB))
    dof = max(1, len(Qdata) - len(res))
    t_critical = t.ppf(1 - alpha / 2, df=dof)
    EB_base = t_critical * SE_base

    eps = 1e-12
    res = np.asarray(res, dtype=float)
    pct_base = 100.0 * EB_base / np.maximum(np.abs(res), eps)

    Sigma_theta_star = CRB
    Cn_Si_hat, Cn_Gr_hat, x_n100_hat, Cp_hat, x_p100_hat, a_hat, b_hat = res

    G = np.zeros((9, 7))
    G[0, :] = [1, 0, 0, 0, 0, 0, 0]  # Cn_Si
    G[1, :] = [0, 1, 0, 0, 0, 0, 0]  # Cn_Gr
    G[2, :] = [0, 0, 0, 1, 0, 0, 0]  # Cp
    G[3, :] = [0, 0, 1, 0, 0, 0, 0]  # x_n100
    G[4, :] = [0, 0, 0, 0, 1, 0, 0]  # x_p100
    G[5, :] = [0, 0, 0, 0, 0, 1, 0]  # silicon OCP voltage scale
    G[6, :] = [0, 0, 0, 0, 0, 0, 1]  # silicon OCP voltage offset
    G[7, :] = [1, 1, 0, 0, 0, 0, 0]  # total anode capacity
    G[8, :] = [x_n100_hat, x_n100_hat, Cn_Si_hat + Cn_Gr_hat, x_p100_hat, Cp_hat, 0, 0]  # lithium inventory

    Sigma_phi = G @ Sigma_theta_star @ G.T
    SE_phi = np.sqrt(np.diag(Sigma_phi))
    EB_phi = t_critical * SE_phi

    phi_hat = np.array([
        Cn_Si_hat,
        Cn_Gr_hat,
        Cp_hat,
        x_n100_hat,
        x_p100_hat,
        a_hat,
        b_hat,
        Cn_Si_hat + Cn_Gr_hat,
        Cp_hat * x_p100_hat + (Cn_Si_hat + Cn_Gr_hat) * x_n100_hat,
    ], dtype=float)

    pct_phi = 100.0 * EB_phi / np.maximum(np.abs(phi_hat), eps)

    return {
        "t_critical": t_critical,
        "base": {"names": BASE_NAMES, "nominal": res.copy(), "EB95": EB_base, "pctErr": pct_base},
        "phi": {"names": PHI_NAMES, "nominal": phi_hat, "EB95": EB_phi, "pctErr": pct_phi},
    }


In [ ]:
plt.rcParams["figure.max_open_warning"] = 0

# Nominal beginning-of-life eSOH vector used as the local linearization point.
# Order saved to CRB tables as [Cn_Si, Cn_Gr, x100, Cp, y100, a, b],
# where x100/y100/a/b correspond to x_n,100/x_p,100/s_V/U_off.


Cn_Si = 1.13
Cn_Gr = 1.465
x100  = 0.9614
Cp    = 2.69
y100  = 0.0173
s_V   = 0.9
U_off = 0.034

res = np.array([Cn_Si, Cn_Gr, x100, Cp, y100, s_V, U_off], dtype=float)
print("Nominal parameter vector:", res)


In [ ]:
# Evaluate one full 0--100% SoC CRB case before the sampling and window sweeps.
Qdata = np.linspace(0, 2.5, 1000)
sigma2 = 9e-6  # voltage-noise variance corresponding to 3 mV bol standard deviation
out_full = crb_and_propagate(
    OCP=OCP,
    compute_jacobian=compute_jacobian,
    res=res,
    Qdata=Qdata,
    sigma2=sigma2,
    alpha=0.05,
    jac_kwargs={"scale": 0.1},
)

print(f"t_critical (95% two-sided) = {out_full['t_critical']:.4f}\n")
print("==== Reported eSOH quantities: propagated CRB ====")
for name, hat, eb, pct in zip(
    out_full["phi"]["names"],
    out_full["phi"]["nominal"],
    out_full["phi"]["EB95"],
    out_full["phi"]["pctErr"],
):
    print(f"{name:8s}: nominal = {hat:.6g} | EB(95%) = {eb:.6g} | error bound = {pct:.2f}%")

print("\n==== Base fitted parameters: direct CRB ====")
for name, hat, eb, pct in zip(
    out_full["base"]["names"],
    out_full["base"]["nominal"],
    out_full["base"]["EB95"],
    out_full["base"]["pctErr"],
):
    print(f"{name:8s}: nominal = {hat:.6g} | EB(95%) = {eb:.6g} | error bound = {pct:.2f}%")


In [ ]:
# Sampling-frequency study: quantify how CRB-derived error bounds decrease as
# more voltage points are sampled over the same 0--100% SoC window.
sigma2 = 9e-6
sample_points = np.logspace(1, 3, num=10, dtype=int)
jac_kwargs = {"scale": 0.1}

percentage_errors_base = {name: [] for name in BASE_NAMES}
error_bounds_base = {name: [] for name in BASE_NAMES}
percentage_errors_phi = {name: [] for name in PHI_NAMES}
error_bounds_phi = {name: [] for name in PHI_NAMES}

for N in sample_points:
    Qdata = np.linspace(0, 2.5, N)
    out = crb_and_propagate(
        OCP=OCP,
        compute_jacobian=compute_jacobian,
        res=res,
        Qdata=Qdata,
        sigma2=sigma2,
        alpha=0.05,
        jac_kwargs=jac_kwargs,
    )

    for name, eb, pct in zip(out["base"]["names"], out["base"]["EB95"], out["base"]["pctErr"]):
        error_bounds_base[name].append(eb)
        percentage_errors_base[name].append(pct)

    for name, eb, pct in zip(out["phi"]["names"], out["phi"]["EB95"], out["phi"]["pctErr"]):
        error_bounds_phi[name].append(eb)
        percentage_errors_phi[name].append(pct)

to_save = {
    "sample_points": sample_points.tolist(),
    "base": {"names": BASE_NAMES, "error_bounds": error_bounds_base, "percentage_errors": percentage_errors_base},
    "phi": {"names": PHI_NAMES, "error_bounds": error_bounds_phi, "percentage_errors": percentage_errors_phi},
}
with open(OUTPUT_DIR / "A03_Sampling_Points_infer.pkl", "wb") as f:
    pickle.dump(to_save, f)


In [ ]:
# Export the sampling-frequency results used in the MATLAB figure script.
sample_points = np.array(sample_points).reshape(-1)
df = pd.DataFrame(percentage_errors_phi)
df.insert(0, "sample_points", sample_points)

sampling_csv = OUTPUT_DIR / "A03_Sampling_Points_Impact_BOL_inferred.csv"
df.to_csv(sampling_csv, index=False)
print("Saved:", sampling_csv)


In [ ]:
# Notebook preview of the sampling-frequency tradeoff. The publication-style
# SVG is generated by F05_2_1_sampling_point.m from the CSV exported above.
custom_styles = {
    "Cn_Si": {"color": "#6600cc", "linestyle": "-", "linewidth": 2, "marker": "o"},
    "Cn_Gr": {"color": "#808080", "linestyle": "-", "linewidth": 2, "marker": "o"},
    "Cp": {"color": "orange", "linestyle": "-", "linewidth": 2, "marker": "o"},
    "x_n100": {"color": "#007acc", "linestyle": "--", "linewidth": 2, "marker": "o"},
    "x_p100": {"color": "orange", "linestyle": "--", "linewidth": 2, "marker": "o"},
    "a": {"color": "#1b9e77", "linestyle": "-", "linewidth": 2, "marker": "o"},
    "b": {"color": "#d95f02", "linestyle": "-", "linewidth": 2, "marker": "o"},
    "Cn": {"color": "#4daf4a", "linestyle": "-", "linewidth": 2, "marker": "o"},
    "LI": {"color": "#e7298a", "linestyle": "-", "linewidth": 2, "marker": "o"},
}

plt.figure(figsize=(10, 6))
for param, style in custom_styles.items():
    plt.plot(sample_points, percentage_errors_phi[param], label=param, **style)

plt.xscale("log")
plt.yscale("log")
plt.xlabel("Number of sampling points")
plt.ylabel("Error bound (%)")
plt.legend()
plt.grid(True, which="both", linestyle="--", color="gray", alpha=0.1)
plt.show()

legacy_sampling_csv = OUTPUT_DIR / "Sampling_Points_Impact_BOL.csv"
pd.DataFrame({**percentage_errors_phi, "point_number": sample_points}).to_csv(legacy_sampling_csv, index=False)
print("Saved:", legacy_sampling_csv)


In [ ]:
# SoC-window coverage study: use 8 points per SoC, so each partial window gets
# a number of samples proportional to its width, then evaluate CRB error bounds.
total_points = 800
jac_kwargs = {"scale": 0.1}
alpha = 0.05

soc_windows = {
    "High SoC": (40, 100),
    "Low SoC": (0, 60),
    "Middle SoC": (20, 80),
    "Deep SoC": (5, 100),
    "Full SoC": (0, 100),
}


def q_range_from_soc_window(low_percent, high_percent, n_points_min):
    """Convert full-cell SoC limits to the corresponding discharge-capacity grid."""
    q_lo = (1 - high_percent / 100.0) * 2.5
    q_hi = (1 - low_percent / 100.0) * 2.5
    n_req = max(n_points_min, len(res) + 5)
    return np.linspace(q_lo, q_hi, n_req)


percentage_errors_varied_phi = {p: {} for p in PHI_NAMES}

for window_name, (low, high) in soc_windows.items():
    window_size = max(1, high - low)
    points_varied = int(round(total_points * (window_size / 100.0)))
    Qdata_varied = q_range_from_soc_window(low, high, points_varied)

    out = crb_and_propagate(
        OCP=OCP,
        compute_jacobian=compute_jacobian,
        res=res,
        Qdata=Qdata_varied,
        sigma2=sigma2,
        alpha=alpha,
        jac_kwargs=jac_kwargs,
    )

    for name, pct in zip(out["phi"]["names"], out["phi"]["pctErr"]):
        percentage_errors_varied_phi[name][window_name] = pct

percentage_errors_varied_df_phi = pd.DataFrame(percentage_errors_varied_phi)
soc_window_csv = OUTPUT_DIR / "A03_SoC_Window_CRB_varied_inferred.csv"
percentage_errors_varied_df_phi.to_csv(soc_window_csv)
print("Saved inferred error bounds by SoC window:", soc_window_csv)
print(percentage_errors_varied_df_phi)


In [ ]:
# Dense lower/upper SoC-window grid used to map valid diagnostic windows.
low_limits = np.linspace(0, 90, 91)
steps = np.arange(10, 101, 1)
n_low, n_steps = len(low_limits), len(steps)

total_points = 800
jac_kwargs = {"scale": 0.1}
alpha = 0.05
percentage_errors_phi = {name: np.full((n_low, n_steps), np.nan, dtype=float) for name in PHI_NAMES}


def q_range_from_soc_window(low_percent, high_percent, n_points_min):
    """Convert lower/upper SoC limits to a capacity grid with positive degrees of freedom."""
    q_lo = (1 - high_percent / 100.0) * 2.5
    q_hi = (1 - low_percent / 100.0) * 2.5
    n_req = max(n_points_min, len(res) + 5)
    return np.linspace(q_lo, q_hi, n_req)


def process_cell(i, j):
    low = float(low_limits[i])
    step = int(steps[j])
    high = low + step
    if high > 100.0:
        return None

    points_varied = int(round(total_points * (step / 100.0)))
    Qdata = q_range_from_soc_window(low, high, points_varied)
    out = crb_and_propagate(
        OCP=OCP,
        compute_jacobian=compute_jacobian,
        res=res,
        Qdata=Qdata,
        sigma2=sigma2,
        alpha=alpha,
        jac_kwargs=jac_kwargs,
    )
    return i, j, dict(zip(out["phi"]["names"], out["phi"]["pctErr"]))


tasks = [(i, j) for i in range(n_low) for j in range(n_steps)]
results = Parallel(n_jobs=-1, prefer="processes")(delayed(process_cell)(i, j) for (i, j) in tasks)

for item in results:
    if item is None:
        continue
    i, j, pct_map = item
    for name, pct in pct_map.items():
        percentage_errors_phi[name][i, j] = pct

out = {
    "phi_names": PHI_NAMES,
    "low_limits": low_limits,
    "steps": steps,
    "percentage_errors_phi": percentage_errors_phi,
}
with open(OUTPUT_DIR / "A03_Incremental_Window_inferred.pkl", "wb") as f:
    pickle.dump(out, f)

print("Saved inferred error-bound grid:", OUTPUT_DIR / "A03_Incremental_Window_inferred.pkl")


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib import rcParams

# Continuous colormap used for lower/upper SoC-window error-bound maps.
dark_hex = "#55748f"
light_hex = "#c36439"
custom_cmap = LinearSegmentedColormap.from_list(
    "blue_to_bronze",
    [mcolors.to_rgb(dark_hex), mcolors.to_rgb(light_hex)],
    N=256,
)
custom_cmap.set_bad(color="white")


def plot_triangle_phi_same_style(
    Z_low_step: np.ndarray,
    low_limits: np.ndarray,
    steps: np.ndarray,
    global_min: float,
    global_max: float,
    title_label: str,
    param_name: str = "param",
    min_window_pct: float = 0,
):
    """Plot CRB error bounds over lower and upper SoC limits.

    The CRB grid is computed as Z(lower SoC, window width). This function
    reprojects it to a triangular lower/upper-SoC canvas so the maps can be used
    to identify whether an available partial SoC window is sufficiently
    informative for each parameter.
    """
    rcParams.update({
        "text.usetex": False,
        "font.family": "sans-serif",
        "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    })

    upper_axis = np.arange(10, 101, 1, dtype=float)
    lower_axis = np.asarray(low_limits, dtype=float)
    extent = [upper_axis.min(), upper_axis.max(), lower_axis.min(), lower_axis.max()]

    Z = np.asarray(Z_low_step, dtype=float)
    M = np.full((len(lower_axis), len(upper_axis)), np.nan, dtype=float)

    for i, low in enumerate(lower_axis):
        uppers = low + steps
        valid = (uppers >= upper_axis.min()) & (uppers <= upper_axis.max())
        cols = (uppers[valid] - upper_axis.min()).astype(int)
        M[i, cols] = Z[i, valid]

    U, L = np.meshgrid(upper_axis, lower_axis)
    geom_mask = U < (L + min_window_pct)
    masked = np.ma.masked_where(geom_mask | (M <= 0) | ~np.isfinite(M), M)

    if (not np.isfinite(global_min)) or (not np.isfinite(global_max)) or (global_min <= 0) or (global_min >= global_max):
        raise ValueError(f"[{param_name}] Invalid LogNorm bounds: vmin={global_min}, vmax={global_max}")
    norm = LogNorm(vmin=global_min, vmax=global_max)

    if np.ma.getmaskarray(masked).all():
        print(f"[{param_name}] all values are masked; no heatmap was saved.")
        return

    fig, ax = plt.subplots(figsize=(5.5, 4))
    im = ax.imshow(
        np.flipud(masked),
        origin="upper",
        extent=extent,
        aspect="auto",
        cmap=custom_cmap,
        norm=norm,
    )

    requested_levels = np.array([1, 2, 5, 10, 15, 50], dtype=float)
    valid_levels = requested_levels[(requested_levels >= global_min) & (requested_levels <= global_max)]
    if valid_levels.size > 0:
        Zc = np.flipud(masked)
        Xc, Yc = np.meshgrid(upper_axis, lower_axis[::-1])
        cs = ax.contour(Xc, Yc, Zc, levels=valid_levels, colors="white", linestyles="solid", linewidths=0.5, norm=norm)
        ax.clabel(cs, inline=True, fmt="%.0f%%", fontsize=16)

    cbar = plt.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=20)
    cbar.set_label("Error bound (log scale)", fontsize=22)

    ax.set_xlabel("Upper SoC limit (%)", fontsize=22)
    ax.set_ylabel("Lower SoC limit (%)", fontsize=22)
    ax.tick_params(axis="both", direction="in", labelsize=20)
    ax.set_title(title_label, fontsize=20)

    for spine in ax.spines.values():
        spine.set_linewidth(0.5)

    plt.tight_layout()
    fig.savefig(FIGURE_DIR / f"A04_SoCwin_{param_name}.svg", format="svg", bbox_inches="tight")
    plt.show()


In [ ]:
# Save triangular SoC-window maps for the reported diagnostic quantities.
PARAMS_TO_PLOT = ["Cn_Si", "Cn_Gr", "Cn", "Cp", "LI", "a", "b"]
TITLE_LABELS = [
    r"$C_{n,\mathrm{Si}}$",
    r"$C_{n,\mathrm{Gr}}$",
    r"$C_n$",
    r"$C_p$",
    r"$LI$",
    r"$s_V$",
    r"$U_{\mathrm{off}}$",
]

pos_vals = []
for p in PARAMS_TO_PLOT:
    arr = np.array(percentage_errors_phi[p], dtype=float)
    pos_vals.append(arr[(arr > 0) & np.isfinite(arr)])
pos = np.concatenate(pos_vals)
global_min, global_max = float(np.nanmin(pos)), float(np.nanmax(pos))

for p, tl in zip(PARAMS_TO_PLOT, TITLE_LABELS):
    print(f"Plotting {p}")
    plot_triangle_phi_same_style(
        Z_low_step=np.asarray(percentage_errors_phi[p]),
        low_limits=np.asarray(low_limits),
        steps=np.asarray(steps),
        global_min=global_min,
        global_max=global_max,
        title_label=tl,
        param_name=p,
        min_window_pct=0,
    )
